# Laboratorio de Cierre Unidad IV: Minería Sujeta a Restricciones en "CineCetys" 🍿🎬

**Contexto del Negocio:**
El Director de Finanzas del cine está furioso. El sistema de recomendación anterior (basado únicamente en Soporte y Confianza) estuvo recomendando combos de productos que tienen un margen de ganancia pésimo o que saturan las manos del cliente impidiendo que los lleve a su asiento.

Te han entregado la base de datos de tickets de la dulcería junto con el **Catálogo de Utilidad Neta** (cuántos pesos libres gana el cine por cada producto tras pagar costos). Tu misión es extraer las reglas de asociación **sujetas a las restricciones estrictas del Director Financiero**.

In [3]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. CATÁLOGO FINANCIERO: Utilidad Neta (Ganancia real en MXN) y Categoría
catalogo_utilidad = {
    'Palomitas Grandes': 65,   # Altísima ganancia (maíz e inflado barato)
    'Refresco Grande': 55,     # Altísima ganancia (jarabe y agua)
    'Nachos con Queso': 45,    # Buena ganancia
    'Hot Dog': 25,             # Ganancia media
    'Chocolate Importado': 8,  # Ganancia pésima (casi todo se paga al proveedor)
    'Agua Embotellada': 12,    # Ganancia baja
    'Helado': 30,              # Ganancia media
    'Boleto VIP': 40,          # Ganancia fija
    'Servilletas Extra': 0     # Costo operativo (Cero ganancia)
}

# 2. HISTORIAL DE TRANSACCIONES EN TAQUILLA/DULCERÍA
tickets_cine = [
    ['Boleto VIP', 'Palomitas Grandes', 'Refresco Grande', 'Servilletas Extra'],
    ['Palomitas Grandes', 'Refresco Grande', 'Chocolate Importado', 'Servilletas Extra'],
    ['Boleto VIP', 'Nachos con Queso', 'Refresco Grande'],
    ['Hot Dog', 'Agua Embotellada', 'Chocolate Importado'],
    ['Palomitas Grandes', 'Refresco Grande', 'Nachos con Queso', 'Servilletas Extra'],
    ['Boleto VIP', 'Palomitas Grandes', 'Refresco Grande', 'Helado'],
    ['Agua Embotellada', 'Chocolate Importado', 'Servilletas Extra'],
    ['Boleto VIP', 'Nachos con Queso', 'Refresco Grande', 'Helado'],
    ['Palomitas Grandes', 'Refresco Grande', 'Hot Dog'],
    ['Boleto VIP', 'Palomitas Grandes', 'Refresco Grande', 'Nachos con Queso']
]

### Reto 1: Restricción Operativa Pre-Apriori (Poda de Basura y Tamaño)

Antes de gastar memoria RAM calculando reglas:
1. El insumo `'Servilletas Extra'` se registra en la caja por control de inventario, pero **no es un producto vendible**. Elimínalo de tu DataFrame booleano antes de correr Apriori (Restricción de Ítem en Pre-procesamiento).
2. Un cliente de cine solo tiene dos manos. La gerencia prohíbe analizar combinaciones de más de **3 artículos en total**. Configura ese límite directamente dentro de la función `apriori()` con un `min_support = 0.2`.

In [ ]:
# ESQUELETO:
# te = TransactionEncoder()
# df_cine = ...
# df_cine_limpio = df_cine.drop(columns=[...])
# itemsets_cine = apriori(..., min_support=0.2, use_colnames=True, max_len=...)
# reglas_cine = association_rules(itemsets_cine, metric="confidence", min_threshold=0.5)

In [7]:
te = TransactionEncoder()
df_cine = pd.DataFrame(te.fit_transform(tickets_cine), columns=te.columns_)

# 1. Eliminamos la columna operativa basura antes de minar
df_cine_limpio = df_cine.drop(columns=['Servilletas Extra'])

# 2. Corremos Apriori restringiendo max_len a 3
itemsets_cine = apriori(df_cine_limpio, min_support=0.2, use_colnames=True, max_len=3)
reglas_cine = association_rules(itemsets_cine, metric="confidence", min_threshold=0.5)


print(f"Reglas generadas tras poda inicial: {len(reglas_cine)}")
display(reglas_cine[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

Reglas generadas tras poda inicial: 31


,antecedents,consequents,support,confidence,lift
0,frozenset({Chocolate Importado}),frozenset({Agua Embotellada}),0.2,0.666667,3.333333
1,frozenset({Agua Embotellada}),frozenset({Chocolate Importado}),0.2,1.000000,3.333333
2,frozenset({Helado}),frozenset({Boleto VIP}),0.2,1.000000,2.000000
3,frozenset({Boleto VIP}),frozenset({Nachos con Queso}),0.3,0.600000,1.500000
4,frozenset({Nachos con Queso}),frozenset({Boleto VIP}),0.3,0.750000,1.500000


### Reto 2: Ingeniería de Restricciones Financieras

El algoritmo Apriori no sabe cuánto dinero deja cada producto. Crea tres columnas nuevas en tu DataFrame `reglas_cine` utilizando el diccionario `catalogo_utilidad`:
1. `utilidad_antecedente`: Suma de la ganancia de los productos en `antecedents`.
2. `utilidad_consecuente`: Suma de la ganancia de los productos en `consequents`.
3. `utilidad_total_combo`: Suma total de la ganancia de toda la regla (`antecedents` + `consequents`).

In [ ]:
# ESQUELETO:
# def obtener_utilidad(conjunto):
#     return sum(catalogo_utilidad[item] for item in conjunto)
#
# reglas_cine['utilidad_antecedente'] = ...
# reglas_cine['utilidad_consecuente'] = ...
# reglas_cine['utilidad_total_combo'] = ...

In [ ]:

def obtener_utilidad(conjunto):
    return sum(catalogo_utilidad[item] for item in conjunto)

reglas_cine['utilidad_antecedente'] = reglas_cine['antecedents'].apply(obtener_utilidad)
reglas_cine['utilidad_consecuente'] = reglas_cine['consequents'].apply(obtener_utilidad)
reglas_cine['utilidad_total_combo'] = reglas_cine['utilidad_antecedente'] + reglas_cine['utilidad_consecuente']

display(reglas_cine[['antecedents', 'consequents', 'confidence', 'utilidad_consecuente', 'utilidad_total_combo']].head())

,antecedents,consequents,confidence,utilidad_consecuente,utilidad_total_combo
0,frozenset({Chocolate Importado}),frozenset({Agua Embotellada}),0.666667,12,20
1,frozenset({Agua Embotellada}),frozenset({Chocolate Importado}),1.000000,8,20
2,frozenset({Helado}),frozenset({Boleto VIP}),1.000000,40,70
3,frozenset({Boleto VIP}),frozenset({Nachos con Queso}),0.600000,45,85
4,frozenset({Nachos con Queso}),frozenset({Boleto VIP}),0.750000,40,85


### Reto 3: El Embudo del Director Financiero (Consulta Sujeta a Restricciones)

Filtra tu tabla `reglas_cine` para entregarle al Director **únicamente** las reglas de oro que cumplan simultáneamente estas 4 restricciones de negocio:
1. **Restricción de Rentabilidad Mínima:** La `utilidad_total_combo` debe ser **mayor o igual a $120 MXN**.
2. **Restricción de Impulso Financiero (Up-selling):** El producto o productos recomendados (`utilidad_consecuente`) deben dejar por sí solos **al menos $45 MXN** de ganancia libre.
3. **Restricción de Ancla:** El antecedente debe incluir forzosamente `'Boleto VIP'` (queremos diseñar promociones exclusivas para cuando el cliente compra su entrada VIP en la app).
4. **Restricción de Calidad Estadística:** El `lift` debe ser estrictamente **mayor a 1.1**.

In [ ]:
# ESQUELETO:
# reglas_oro_vip = reglas_cine[
#     (Condicion 1) &
#     (Condicion 2) &
#     (Condicion 3: usa .apply(lambda x: 'Boleto VIP' in x)) &
#     (Condicion 4)
# ]

In [ ]:
reglas_oro_vip = reglas_cine[
    (reglas_cine['utilidad_total_combo'] >= 120) &
    (reglas_cine['utilidad_consecuente'] >= 45) &
    (reglas_cine['antecedents'].apply(lambda x: 'Boleto VIP' in x)) &
    (reglas_cine['lift'] > 1.1)
].sort_values(by='utilidad_total_combo', ascending=False)

print("--- REGLAS DE ORO APROBADAS POR FINANZAS ---")
display(reglas_oro_vip[['antecedents', 'consequents', 'confidence', 'lift', 'utilidad_consecuente', 'utilidad_total_combo']])

--- REGLAS DE ORO APROBADAS POR FINANZAS ---


,antecedents,consequents,confidence,lift,utilidad_consecuente,utilidad_total_combo
24,"frozenset({Boleto VIP, Palomitas Grandes})",frozenset({Refresco Grande}),1.0,1.25,55,160
19,"frozenset({Boleto VIP, Refresco Grande})",frozenset({Nachos con Queso}),0.6,1.50,45,140
18,"frozenset({Boleto VIP, Nachos con Queso})",frozenset({Refresco Grande}),1.0,1.25,55,140
21,frozenset({Boleto VIP}),"frozenset({Nachos con Queso, Refresco Grande})",0.6,1.50,100,140
15,"frozenset({Boleto VIP, Helado})",frozenset({Refresco Grande}),1.0,1.25,55,125


### 🧠 Reto 4: Defensa de Ingeniería y Optimización (Responde aquí)

**1. Pre-poda vs Post-filtración (Arquitectura Computacional):**
En el Reto 1 borramos `'Servilletas Extra'` **antes** de ejecutar `apriori()`, pero en el Reto 3 filtramos la utilidad de `$120 MXN` **después** de ejecutar `association_rules()`. 
¿Qué habría pasado con el tiempo de cómputo y la memoria RAM si en una base de datos de 10 millones de tickets hubiéramos dejado las servilletas adentro y las hubiéramos filtrado hasta el final en el Reto 3?
> **Tu Respuesta:** [Escribe tu justificación aquí]

**2. La Trampa del "Chocolate Importado":**
Si revisas los tickets crudos, la gente que compra `Agua Embotellada` casi siempre compra `Chocolate Importado` (tienen alta Confianza y alto Lift). ¿Por qué nuestro sistema de restricciones eliminó por completo esa regla y por qué eso salvó a la empresa de un error de marketing?
> **Tu Respuesta:** [Escribe tu justificación aquí]

In [ ]:
import random
import math

def generar_equipos(lista_alumnos, group_size = 5):
    alumnos_mezclados = lista_alumnos[:]
    random.shuffle(alumnos_mezclados)
    
    n = len(alumnos_mezclados)
    if n == 0:
        return []
        
    num_grupos = math.ceil(n / group_size)
    
    tamanos_grupos = [n // num_grupos + (1 if i < n % num_grupos else 0) for i in range(num_grupos)]
    
    equipos = []
    indice = 0
    
    for tamano in tamanos_grupos:
        equipos.append(alumnos_mezclados[indice:indice + tamano])
        indice += tamano
        
    return equipos
#"Linette","Angela",
alumnos = [ "Rogelio", "Juan Pablo", "Octavio", "Cristobal",
"Ramiro", "Diego Angel", "Emiliano", "Carlos Miguel",
"Andrea", "Rhamses", "Mayrin", "Christopher", "Kamila","Andres", "Diego Torres",
"Raul", "Ximena", "Luis", "Diego Lopez", "Juan Manuel", "Erik"]
equipos_formados = generar_equipos(alumnos,2)

for i, equipo in enumerate(equipos_formados, 1):
    print(f"Equipo {i} ({len(equipo)} integrantes): {', '.join(equipo)}")